In [2]:
"""
Week 6 - Function 1
True Bayesian NN surrogate with Bayesian linear head (Pyro VI)
using the supplied 2D inputs and 1D outputs.

Requirements (local):
    pip install numpy torch pyro-ppl scikit-learn
"""

import math
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


# ------------------------ 1. Config ------------------------

RANDOM_SEED = 123
INPUT_DIM = 2              # Function 1 is 2D
HIDDEN_SIZES = [64, 64]    # MLP feature extractor
N_EPOCHS_FEATURE = 2000    # epochs for deterministic pretraining
LR_FEATURE = 1e-3

N_STEPS_VI = 3000          # SVI steps for Bayesian head
LR_VI = 5e-3
PRIOR_SCALE = 1.0          # prior std for weights / bias

N_CANDIDATES = 20000       # random candidate points in [0, 1]^2
N_PRED_SAMPLES = 200       # posterior predictive samples
XI = 0.01                  # exploration parameter for EI (in scaled y-space)
TOP_K = 5                  # how many top candidates to print


# ------------------------ 2. Data ------------------------

def load_data():
    """
    Uses the supplied 2D inputs and 1D outputs for Week6 - Function 1.
    X_raw: shape (14, 2)
    y_raw: shape (14,)
    """

    X_raw = np.array([
        [0.31940389, 0.76295937],
        [0.57432921, 0.87989810],
        [0.73102363, 0.73299988],
        [0.84035342, 0.26473161],
        [0.65011406, 0.68152635],
        [0.41043714, 0.14755430],
        [0.31269116, 0.07872278],
        [0.68341817, 0.86105746],
        [0.08250725, 0.40348751],
        [0.88388983, 0.58225397],
        [0.88389,    0.98389],
        [0.37454,    0.950713],
        [0.382224,   0.951319],
        [0.782778,   0.793329],
        [0.030500,   0.037300],
        [0.646168,   0.172681],
    ], dtype=np.float64)

    y_raw = np.array([
        1.32267704e-79,
        1.03307824e-46,
        7.71087511e-16,
        3.34177101e-124,
        -3.60606264e-03,
        -2.15924904e-54,
        -2.08909327e-91,
        2.53500115e-40,
        3.60677119e-81,
        6.22985647e-48,
        9.59033053e-135,
        -1.56227724e-117,
        -4.77166224e-115,
        7.535209723645751e-36,
        1.6357533426693436e-209,
        6.327028545366271e-79,
    ], dtype=np.float64)

    assert X_raw.shape[1] == INPUT_DIM, "INPUT_DIM must match your data dimension."
    return X_raw, y_raw


# ------------------------ 3. Feature extractor (deterministic) ------------------------

class FeatureExtractor(nn.Module):
    """
    Simple MLP that maps x in R^2 to a hidden feature vector.
    We'll freeze this after pretraining.
    """
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        self.net = nn.Sequential(*layers)
        self.output_dim = prev_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DeterministicRegressor(nn.Module):
    """
    MLP regressor used to pretrain the feature extractor.
    feature_extractor -> linear head -> scalar output
    """
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        self.feature_extractor = FeatureExtractor(input_dim, hidden_sizes)
        self.head = nn.Linear(self.feature_extractor.output_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.feature_extractor(x)
        return self.head(feats).squeeze(-1)


def train_feature_extractor(X: torch.Tensor,
                            y: torch.Tensor) -> FeatureExtractor:
    """
    Train the deterministic regressor to fit (X, y) with MSE.
    Returns the trained feature extractor (weights frozen later).
    """
    torch.manual_seed(RANDOM_SEED)
    model = DeterministicRegressor(INPUT_DIM, HIDDEN_SIZES)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_FEATURE)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(N_EPOCHS_FEATURE):
        optimizer.zero_grad()
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

    # Extract trained feature extractor
    feature_extractor = FeatureExtractor(INPUT_DIM, HIDDEN_SIZES)
    feature_extractor.load_state_dict(model.feature_extractor.state_dict())
    return feature_extractor


# ------------------------ 4. Bayesian linear head (Pyro model) ------------------------

@dataclass
class BayesianHeadConfig:
    prior_scale: float = PRIOR_SCALE


def make_bayesian_model(feature_extractor: FeatureExtractor,
                        cfg: BayesianHeadConfig):
    """
    Returns a Pyro model that uses the given (frozen) feature_extractor
    and a Bayesian linear head (weights, bias, noise sigma).
    """

    def model(x, y=None):
        # Freeze the feature extractor parameters (point estimates)
        pyro.module("feature_extractor", feature_extractor, update_module_params=False)

        # x: (N, D)
        feats = feature_extractor(x)              # (N, H)
        H = feats.size(-1)

        # Priors over the linear head parameters
        weight = pyro.sample(
            "weight",
            dist.Normal(x.new_zeros(H), cfg.prior_scale * x.new_ones(H)).to_event(1)
        )  # shape (H,)

        bias = pyro.sample(
            "bias",
            dist.Normal(x.new_tensor(0.0), cfg.prior_scale)
        )  # scalar

        # Noise scale (likelihood std)
        sigma = pyro.sample(
            "sigma",
            dist.HalfCauchy(x.new_tensor(0.1))
        )

        mean = (feats * weight).sum(dim=-1) + bias  # (N,)

        with pyro.plate("data", x.size(0)):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

    return model


def train_bayesian_head(model, X: torch.Tensor, y: torch.Tensor):
    """
    Run SVI to approximate the posterior of the Bayesian linear head.
    Returns (guide, svi_loss_history).
    """
    pyro.clear_param_store()
    guide = AutoDiagonalNormal(model)
    optimizer = Adam({"lr": LR_VI})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

    loss_history = []
    for step in range(N_STEPS_VI):
        loss = svi.step(X, y)
        loss_history.append(loss)

    return guide, loss_history


# ------------------------ 5. Acquisition functions ------------------------

def normal_cdf(x: torch.Tensor) -> torch.Tensor:
    """Standard normal CDF."""
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


def normal_pdf(x: torch.Tensor) -> torch.Tensor:
    """Standard normal PDF."""
    return (1.0 / math.sqrt(2.0 * math.pi)) * torch.exp(-0.5 * x**2)


def compute_ei_and_pi(samples: torch.Tensor,
                      best_y: float,
                      xi: float = XI):
    """
    samples: (n_samples, n_points) predictive samples in *scaled* y-space
    best_y: scalar, current best in scaled space
    Returns:
      ei: (n_points,)
      pi: (n_points,)
      mean: (n_points,)
      std: (n_points,)
    """
    # Predictive mean and std from Monte Carlo samples
    mean = samples.mean(dim=0)     # (n_points,)
    std = samples.std(dim=0) + 1e-9

    # EI (using analytic normal approx from mean/std)
    gamma = (mean - best_y - xi) / std
    ei = (mean - best_y - xi) * normal_cdf(gamma) + std * normal_pdf(gamma)
    ei = torch.clamp(ei, min=0.0)

    # Probability of Improvement (PI)
    pi = normal_cdf((mean - best_y - xi) / std)

    return ei, pi, mean, std


# ------------------------ 6. Propose next point ------------------------

def propose_next_point(model,
                       guide,
                       feature_extractor: FeatureExtractor,
                       X_train_scaled: torch.Tensor,
                       y_train_scaled: torch.Tensor,
                       scaler_y: StandardScaler):
    """
    Draw posterior predictive samples on random candidates in [0, 1]^2,
    compute EI, and propose next query point.

    Returns:
        next_x: np.ndarray (2,)
        info: dict with details and top-k candidates
    """
    # Generate random candidate points in [0, 1]^2
    X_candidates = torch.rand((N_CANDIDATES, INPUT_DIM))

    # Predictive distribution via Pyro Predictive
    predictive = Predictive(
        model,
        guide=guide,
        num_samples=N_PRED_SAMPLES,
        return_sites=("obs",)
    )

    with torch.no_grad():
        pred = predictive(X_candidates)  # obs: (n_samples, n_points)
        samples_scaled = pred["obs"]     # shape (S, N)

    best_y_scaled = float(y_train_scaled.max().item())

    ei, pi, mean_scaled, std_scaled = compute_ei_and_pi(
        samples_scaled, best_y_scaled, xi=XI
    )

    # Choose candidate with maximum EI
    idx_best = int(torch.argmax(ei).item())
    next_x = X_candidates[idx_best].cpu().numpy()

    # Convert mean/std back to original y-scale for reporting
    mean_scaled_np = mean_scaled.cpu().numpy()
    std_scaled_np = std_scaled.cpu().numpy()

    mean_raw = scaler_y.inverse_transform(
        mean_scaled_np.reshape(-1, 1)
    ).squeeze(-1)                               # shape (N,)
    std_raw = std_scaled_np * scaler_y.scale_[0]

    # Current best in original scale
    best_y_raw_arr = scaler_y.inverse_transform(
        np.array(best_y_scaled).reshape(-1, 1)
    )
    best_y_raw = float(best_y_raw_arr.squeeze())

    # Prepare top-k candidates for reporting (cast to Python floats)
    topk_vals, topk_idx = torch.topk(ei, k=min(TOP_K, N_CANDIDATES))

    top_candidates = []
    for rank, (ei_val, j) in enumerate(zip(topk_vals, topk_idx)):
        j = int(j.item())
        x_j = X_candidates[j].cpu().numpy()
        mean_j = float(mean_raw[j])
        std_j = float(std_raw[j])
        pi_j = float(pi[j].item())
        top_candidates.append({
            "rank": rank + 1,
            "x": x_j,
            "ei_scaled": float(ei_val.item()),
            "pred_mean": mean_j,
            "pred_std": std_j,
            "prob_improvement": pi_j,
        })

    info = {
        "next_x": next_x,
        "next_pred_mean": float(mean_raw[idx_best]),
        "next_pred_std": float(std_raw[idx_best]),
        "next_prob_improvement": float(pi[idx_best].item()),
        "best_y_raw": float(best_y_raw),
        "top_candidates": top_candidates,
    }

    return next_x, info


# ------------------------ 7. Main pipeline ------------------------

def main():
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    # 1) Load data
    X_raw, y_raw = load_data()
    print(f"Loaded X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")

    # 2) Scale outputs (inputs are already 0..1 so we keep them as-is)
    scaler_y = StandardScaler()
    y_scaled_np = scaler_y.fit_transform(y_raw.reshape(-1, 1)).astype(np.float32).ravel()

    X_t = torch.from_numpy(X_raw.astype(np.float32))
    y_t = torch.from_numpy(y_scaled_np.astype(np.float32))

    # 3) Pre-train deterministic MLP feature extractor
    print("=== Pretraining deterministic feature extractor ===")
    feature_extractor = train_feature_extractor(X_t, y_t)
    feature_extractor.eval()
    print("Feature extractor trained.\n")

    # 4) Build Bayesian model (feature extractor + Bayesian linear head)
    print("=== Training Bayesian linear head (VI, Pyro) ===")
    bayes_cfg = BayesianHeadConfig(prior_scale=PRIOR_SCALE)
    bayes_model = make_bayesian_model(feature_extractor, bayes_cfg)

    guide, loss_history = train_bayesian_head(bayes_model, X_t, y_t)
    print("Bayesian head training complete.\n")

    # 5) Propose next query point via EI
    print("=== Proposing next query point (EI with Bayesian NN) ===")
    next_x, info = propose_next_point(
        bayes_model,
        guide,
        feature_extractor,
        X_t,
        y_t,
        scaler_y
    )

    # Report results
    print("\n=== CURRENT BEST (from observed data) ===")
    print(f"Best observed y (original scale): {info['best_y_raw']:.6g}")

    print("\n=== PROPOSED NEXT QUERY POINT ===")
    print(f"x_next (in [0,1]^2): {info['next_x']}")
    print(f"Predicted y at x_next (mean, original scale): {info['next_pred_mean']:.6g}")
    print(f"Predictive std at x_next (original scale): {info['next_pred_std']:.6g}")
    print(f"Probability of improvement over current best: "
          f"{info['next_prob_improvement'] * 100:.2f}%")

    print("\n=== TOP CANDIDATES (by EI, in [0,1]^2) ===")
    for cand in info["top_candidates"]:
        x = cand["x"]
        print(
            f"Rank {cand['rank']:>2d}: x={x}, "
            f"pred_mean={cand['pred_mean']:.6g}, "
            f"pred_std={cand['pred_std']:.6g}, "
            f"PI={cand['prob_improvement']*100:5.2f}%, "
            f"EI_scaled={cand['ei_scaled']:.4g}"
        )

    print("\nDone.")


if __name__ == "__main__":
    main()


Loaded X_raw shape: (14, 2), y_raw shape: (14,)
=== Pretraining deterministic feature extractor ===
Feature extractor trained.

=== Training Bayesian linear head (VI, Pyro) ===
Bayesian head training complete.

=== Proposing next query point (EI with Bayesian NN) ===

=== CURRENT BEST (from observed data) ===
Best observed y (original scale): -2.02348e-13

=== PROPOSED NEXT QUERY POINT ===
x_next (in [0,1]^2): [0.64616853 0.17268163]
Predicted y at x_next (mean, original scale): 0.000104698
Predictive std at x_next (original scale): 0.00019236
Probability of improvement over current best: 69.01%

=== TOP CANDIDATES (by EI, in [0,1]^2) ===
Rank  1: x=[0.64616853 0.17268163], pred_mean=0.000104698, pred_std=0.00019236, PI=69.01%, EI_scaled=0.144
Rank  2: x=[0.66758966 0.19184858], pred_mean=9.74817e-05, pred_std=0.000195478, PI=67.41%, EI_scaled=0.1399
Rank  3: x=[0.6208145 0.1682995], pred_mean=0.000102141, pred_std=0.000183816, PI=69.33%, EI_scaled=0.1388
Rank  4: x=[0.646981   0.15354